In [ ]:
"Shor's Algorithm - Scaling Analysis for Classical Period Finding"
import math
import numpy as np
import time
import random
import matplotlib.pyplot as plt

# ═══════════════════════════════════════════════════════
#  Core mathematical functions
# ═══════════════════════════════════════════════════════

def gcd(a, b):
    """Euclidean algorithm for greatest common divisor."""
    while b:
        a, b = b, a % b
    return a

def is_prime(n):
    """Trial division primality test."""
    if n < 2 or (n > 2 and n % 2 == 0):
        return False
    if n == 2:
        return True
    for i in range(3, int(math.sqrt(n)) + 1, 2):
        if n % i == 0:
            return False
    return True

def factorize(N):
    """Return list of prime factors (trial division)."""
    factors = []
    d, temp = 2, N
    while d * d <= temp:
        while temp % d == 0:
            factors.append(d)
            temp //= d
        d += 1
    if temp > 1:
        factors.append(temp)
    return factors

def is_semiprime(n):
    """True if n = p × q for distinct primes (RSA-style)."""
    f = factorize(n)
    return len(f) == 2 and f[0] != f[1]

# ═══════════════════════════════════════════════════════
#  Shor's algorithm (classical simulation)
# ═══════════════════════════════════════════════════════

def classical_order(x, N):
    """Find smallest r with x^r ≡ 1 (mod N). Returns (r, op_count, time_seconds)."""
    r, value, ops = 1, x % N, 0
    start = time.perf_counter()
    while value != 1:
        value = (value * x) % N
        r, ops = r + 1, ops + 1
        if r > N:
            elapsed = time.perf_counter() - start
            return None, ops, elapsed
    elapsed = time.perf_counter() - start
    return r, ops, elapsed

def analyze_single_iteration(N, a):
    """One Shor iteration: pick base a, find order, extract factors if possible."""
    ops = 1
    period_time = 0.0
    if gcd(a, N) != 1:  # Lucky: a shares factor with N
        return {'success': True, 'operations': ops, 'period_time': period_time}
    r, o, period_time = classical_order(a, N)
    ops += o
    if r is None or r % 2 != 0:  # Need even order
        return {'success': False, 'operations': ops, 'period_time': period_time}
    x_half = pow(a, r // 2, N)
    if x_half == N - 1:  # x^(r/2) ≡ -1: no useful factors
        return {'success': False, 'operations': ops + 1, 'period_time': period_time}
    p, q = gcd(x_half - 1, N), gcd(x_half + 1, N)  # Extract factors
    success = (1 < p < N) or (1 < q < N)
    return {'success': success, 'operations': ops + 3, 'period_time': period_time}

def run_iterations(N, num_iterations=10):
    """Run Shor iterations with random coprime bases; return stats with per-iteration data."""
    bases = [a for a in range(2, min(N, 10000)) if gcd(a, N) == 1]
    if not bases:
        return None
    
    # Warm-up run to avoid JIT/startup overhead skewing first timed iteration
    _ = analyze_single_iteration(N, random.choice(bases))
    
    success_count = 0
    ops_list = []
    period_time_list = []
    
    for _ in range(num_iterations):
        r = analyze_single_iteration(N, random.choice(bases))
        ops_list.append(r['operations'])
        period_time_list.append(r['period_time'] * 1000)  # Convert to ms
        if r['success']:
            success_count += 1
    
    ops_arr = np.array(ops_list)
    period_time_arr = np.array(period_time_list)
    
    return {
        'N': N, 'num_bits': N.bit_length(), 'num_digits': len(str(N)),
        'factors': factorize(N), 'num_iterations': num_iterations,
        'success_count': success_count, 'success_rate': 100 * success_count / num_iterations,
        'total_operations': int(ops_arr.sum()),
        'avg_operations': ops_arr.mean(),
        'std_operations': ops_arr.std(),
        'min_operations': int(ops_arr.min()),
        'max_operations': int(ops_arr.max()),
        'avg_period_time_ms': period_time_arr.mean(),
        'std_period_time_ms': period_time_arr.std(),
    }

# ═══════════════════════════════════════════════════════
#  Semiprime generation
# ═══════════════════════════════════════════════════════

def generate_semiprimes(min_n, max_n, num_samples):
    """Equally spaced semiprimes in [min_n, max_n]."""
    targets = np.linspace(min_n, max_n, num_samples)
    numbers = []
    for t in targets:
        start = max(15, int(t) | 1)  # Nearest odd ≥ 15
        for i in range(5000):  # Search outward for semiprime
            for c in [start + 2*i, max(15, start - 2*i)]:
                if is_semiprime(c):
                    numbers.append(c)
                    break
            else:
                continue
            break
        else:
            numbers.append(15)
    return sorted(set(numbers))

# ═══════════════════════════════════════════════════════
#  Extrapolation
# ═══════════════════════════════════════════════════════

def extrapolate(results):
    """Fit ops = a×N^b; extrapolate to RSA key sizes."""
    N_arr = np.array([r['N'] for r in results])
    log_N, log_ops = np.log10(N_arr), np.log10(np.array([r['avg_operations'] for r in results]))
    slope, intercept = np.polyfit(log_N, log_ops, 1)
    a = 10**intercept
    r2 = 1 - np.sum((log_ops - (slope*log_N + intercept))**2) / np.sum((log_ops - np.mean(log_ops))**2)
    
    ref = results[-1]
    ref_time, ref_ops = ref['avg_period_time_ms']/1000, ref['avg_operations']
    sec_per_year = 365.25 * 24 * 3600
    rsa = {}
    for name, bits in [('RSA-512', 512), ('RSA-1024', 1024), ('RSA-2048', 2048), ('RSA-4096', 4096)]:
        log_N_rsa = bits * np.log10(2)
        log_ops_rsa = intercept + slope * log_N_rsa
        log_time = log_ops_rsa + np.log10(ref_time) - np.log10(ref_ops) if ref_ops else -9
        log_years = log_time - np.log10(sec_per_year)
        rsa[name] = {'ops': log_ops_rsa, 'years': log_years}
    
    return {'exponent': slope, 'coefficient': a, 'log_a': intercept, 'r_squared': r2, 'rsa': rsa,
            'fitted_log_ops': intercept + slope * log_N, 'log_N': log_N, 'log_ops': log_ops}

# ═══════════════════════════════════════════════════════
#  Plotting
# ═══════════════════════════════════════════════════════

def plot_results(results, fit):
    """Generate all plots for the report."""
    log_N = fit['log_N']
    log_ops = fit['log_ops']
    fitted = fit['fitted_log_ops']
    residuals = log_ops - fitted
    
    N_arr = np.array([r['N'] for r in results])
    avg_ops = np.array([r['avg_operations'] for r in results])
    std_ops = np.array([r['std_operations'] for r in results])
    success_rates = np.array([r['success_rate'] for r in results])
    avg_period_times = np.array([r['avg_period_time_ms'] for r in results])
    std_period_times = np.array([r['std_period_time_ms'] for r in results])
    bits_arr = np.array([r['num_bits'] for r in results])
    
    # Convert std_ops to log-space error bars (asymmetric) — clamp to avoid negatives
    log_ops_upper = np.maximum(np.log10(avg_ops + std_ops) - log_ops, 0)
    lower_bound_ops = np.maximum(avg_ops - std_ops, avg_ops * 0.01)
    log_ops_lower = np.maximum(log_ops - np.log10(lower_bound_ops), 0)
    
    # ── Plot 1: Log-log scaling with fitted line and error bars ──
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.errorbar(log_N, log_ops, yerr=[log_ops_lower, log_ops_upper],
                fmt='o', markersize=3, alpha=0.4, color='steelblue', ecolor='lightsteelblue',
                elinewidth=0.8, capsize=0, label='Measured data (mean ± 1σ)')
    ax.plot(log_N, fitted, color='red', linewidth=2,
            label=f'Power law fit: ops = {fit["coefficient"]:.4f} × N$^{{{fit["exponent"]:.3f}}}$\n'
                  f'R² = {fit["r_squared"]:.4f}')
    ax.set_xlabel('log₁₀(N) [semiprime value]', fontsize=11)
    ax.set_ylabel('log₁₀(Operations) [modular multiplications]', fontsize=11)
    ax.set_title('Classical Period Finding: Operation Count Scaling (Log-Log)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()
    
    # ── Plot 2: Linear scale — Avg operations vs N with error bars ──
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.errorbar(N_arr, avg_ops, yerr=std_ops,
                fmt='o', markersize=3, alpha=0.4, color='steelblue', ecolor='lightsteelblue',
                elinewidth=0.8, capsize=0, label='Measured data (mean ± 1σ)')
    N_fit_line = np.linspace(N_arr[0], N_arr[-1], 200)
    ops_fit_line = fit['coefficient'] * N_fit_line ** fit['exponent']
    ax.plot(N_fit_line, ops_fit_line, color='red', linewidth=2,
            label=f'Power law fit: ops = {fit["coefficient"]:.4f} × N$^{{{fit["exponent"]:.3f}}}$')
    ax.set_xlabel('N [semiprime value]', fontsize=11)
    ax.set_ylabel('Avg Operations [modular multiplications]', fontsize=11)
    ax.set_title('Classical Period Finding: Operation Count Scaling (Linear)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # ── Plot 3: Residuals — fit quality diagnostic ──
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.scatter(log_N, residuals, s=10, alpha=0.5, color='steelblue', label='Residuals')
    ax.axhline(y=0, color='red', linewidth=1.5, linestyle='--', label='Zero line')
    ax.fill_between(log_N, -np.std(residuals), np.std(residuals),
                    alpha=0.1, color='red', label=f'±1σ band (σ = {np.std(residuals):.3f})')
    ax.set_xlabel('log₁₀(N) [semiprime value]', fontsize=11)
    ax.set_ylabel('Residual [log₁₀(measured) − log₁₀(fitted)]', fontsize=11)
    ax.set_title(f'Power Law Fit Residuals — Mean: {np.mean(residuals):.4f}, Std Dev: {np.std(residuals):.4f}',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # ── Plot 4: Log-log scaling with RSA extrapolation ──
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.scatter(log_N, log_ops, s=8, alpha=0.4, color='steelblue', label='Measured data')
    rsa_bits = [512, 1024, 2048, 4096]
    rsa_log_N = [b * np.log10(2) for b in rsa_bits]
    rsa_log_ops = [fit['rsa'][f'RSA-{b}']['ops'] for b in rsa_bits]
    extend_x = np.linspace(log_N[0], max(rsa_log_N), 200)
    extend_y = fit['log_a'] + fit['exponent'] * extend_x
    ax.plot(extend_x, extend_y, color='red', linewidth=1.5, linestyle='--',
            label=f'Power law extrapolation (exponent = {fit["exponent"]:.3f})')
    ax.scatter(rsa_log_N, rsa_log_ops, s=120, color='red', marker='*', zorder=5,
              label='RSA key sizes (extrapolated)')
    for bits, lx, ly in zip(rsa_bits, rsa_log_N, rsa_log_ops):
        years = fit['rsa'][f'RSA-{bits}']['years']
        ax.annotate(f'RSA-{bits}\n≈ 10$^{{{ly:.0f}}}$ ops\n≈ 10$^{{{years:.0f}}}$ years',
                    (lx, ly), textcoords='offset points', xytext=(12, -20),
                    fontsize=8, color='darkred',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray', alpha=0.8))
    ax.set_xlabel('log₁₀(N) [semiprime value]', fontsize=11)
    ax.set_ylabel('log₁₀(Operations) [modular multiplications]', fontsize=11)
    ax.set_title('Classical Period Finding: Extrapolation to RSA Key Sizes', fontsize=13, fontweight='bold')
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # ── Plot 5: Period-finding execution time scaling with error bars ──
    valid_mask = avg_period_times > 0
    log_N_valid = log_N[valid_mask]
    avg_pt_valid = avg_period_times[valid_mask]
    std_pt_valid = std_period_times[valid_mask]
    log_period_time = np.log10(avg_pt_valid / 1000)  # ms to seconds
    # Asymmetric error bars in log space — clamp to avoid negative yerr
    log_pt_upper = np.maximum(np.log10((avg_pt_valid + std_pt_valid) / 1000) - log_period_time, 0)
    # Clamp lower bound: avg-std must stay positive for valid log
    lower_bound = np.maximum(avg_pt_valid - std_pt_valid, avg_pt_valid * 0.01)
    log_pt_lower = np.maximum(log_period_time - np.log10(lower_bound / 1000), 0)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.errorbar(log_N_valid, log_period_time, yerr=[log_pt_lower, log_pt_upper],
                fmt='o', markersize=3, alpha=0.4, color='darkorange', ecolor='navajowhite',
                elinewidth=0.8, capsize=0, label='Measured period-finding time (mean ± 1σ)')
    time_slope, time_intercept = np.polyfit(log_N_valid, log_period_time, 1)
    ax.plot(log_N_valid, time_intercept + time_slope * log_N_valid, color='red', linewidth=2,
            label=f'Power law fit: time ∝ N$^{{{time_slope:.3f}}}$')
    ax.set_xlabel('log₁₀(N) [semiprime value]', fontsize=11)
    ax.set_ylabel('log₁₀(Period-Finding Time) [seconds]', fontsize=11)
    ax.set_title('Classical Period Finding: Execution Time Scaling', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, which='both')
    plt.tight_layout()
    plt.show()
    
    # ── Plot 6: Period-finding time vs operations (linearity check) ──
    valid_ops = avg_ops[valid_mask]
    valid_times = avg_pt_valid
    std_times_valid = std_pt_valid
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.errorbar(valid_ops, valid_times, yerr=std_times_valid,
                fmt='o', markersize=3, alpha=0.4, color='purple', ecolor='plum',
                elinewidth=0.8, capsize=0, label='Measured data (mean ± 1σ)')
    # Fit line through origin: time = k * ops
    k = np.sum(valid_ops * valid_times) / np.sum(valid_ops ** 2)
    ops_line = np.linspace(0, valid_ops.max(), 200)
    ax.plot(ops_line, k * ops_line, color='red', linewidth=2,
            label=f'Linear fit: time ≈ {k:.6f} ms per operation')
    ax.set_xlabel('Avg Operations [modular multiplications]', fontsize=11)
    ax.set_ylabel('Avg Period-Finding Time [ms]', fontsize=11)
    ax.set_title('Period-Finding Time vs Operation Count (Linearity Validation)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # ── Plot 7: Success rate vs N ──
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(log_N, success_rates, s=12, alpha=0.5, color='green', label='Per-semiprime success rate')
    avg_success = np.mean(success_rates)
    ax.axhline(y=avg_success, color='red', linestyle='--', linewidth=1.5,
               label=f'Mean success rate: {avg_success:.1f}%')
    ax.set_xlabel('log₁₀(N) [semiprime value]', fontsize=11)
    ax.set_ylabel('Success Rate [%]', fontsize=11)
    ax.set_title("Shor's Algorithm: Factoring Success Rate Across Semiprime Range",
                 fontsize=13, fontweight='bold')
    ax.set_ylim(-5, 105)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # ── Plot 8: Operations vs bit length (exponential in bits) ──
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.errorbar(bits_arr, log_ops, yerr=[log_ops_lower, log_ops_upper],
                fmt='o', markersize=3, alpha=0.4, color='steelblue', ecolor='lightsteelblue',
                elinewidth=0.8, capsize=0, label='Measured data (mean ± 1σ)')
    bit_slope, bit_intercept = np.polyfit(bits_arr, log_ops, 1)
    bits_line = np.linspace(bits_arr.min(), bits_arr.max(), 200)
    ax.plot(bits_line, bit_intercept + bit_slope * bits_line, color='red', linewidth=2,
            label=f'Linear fit: log₁₀(ops) = {bit_slope:.3f} × bits + ({bit_intercept:.3f})\n'
                  f'→ operations grow exponentially with bit length')
    ax.set_xlabel('Bit Length of N [bits]', fontsize=11)
    ax.set_ylabel('log₁₀(Operations) [modular multiplications]', fontsize=11)
    ax.set_title('Classical Period Finding: Operations vs Bit Length (Exponential Growth)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # ── Plot 9: RSA extrapolation bar chart ──
    fig, ax = plt.subplots(figsize=(8, 5))
    rsa_names = list(fit['rsa'].keys())
    rsa_years = [fit['rsa'][k]['years'] for k in rsa_names]
    rsa_ops_vals = [fit['rsa'][k]['ops'] for k in rsa_names]
    bars = ax.bar(rsa_names, rsa_years, color=['#3498db', '#e74c3c', '#2ecc71', '#9b59b6'],
                  edgecolor='black', linewidth=0.5)
    ax.set_xlabel('RSA Key Size', fontsize=11)
    ax.set_ylabel('log₁₀(Estimated Time) [years]', fontsize=11)
    ax.set_title('Estimated Classical Factoring Time for RSA Key Sizes',
                 fontsize=13, fontweight='bold')
    for bar, years_val, ops_val in zip(bars, rsa_years, rsa_ops_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
                f'≈ 10$^{{{years_val:.0f}}}$ years\n(10$^{{{ops_val:.0f}}}$ ops)',
                ha='center', fontsize=9, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

# ═══════════════════════════════════════════════════════
#  Main execution
# ═══════════════════════════════════════════════════════

def main():
    MIN_N, MAX_N, NUM_SAMPLES, ITERATIONS = 15, 100_000_000, 1000, 10
    
    N_values = generate_semiprimes(MIN_N, MAX_N, NUM_SAMPLES)
    results = []
    start = time.time()
    
    for i, N in enumerate(N_values, 1):
        if i % 10 == 0 or i == 1:
            pct = 100 * i / len(N_values)
            eta = (time.time() - start) / i * (len(N_values) - i) if i else 0
            print(f"[{i}/{len(N_values)}] {pct:.1f}% | ETA: {eta:.0f}s")
        r = run_iterations(N, ITERATIONS)
        if r:
            results.append(r)
    
    fit = extrapolate(results)
    elapsed = time.time() - start
    print(f"\nDone in {elapsed:.1f}s")
    print(f"Scaling: O(N^{fit['exponent']:.2f}), R² = {fit['r_squared']:.3f}")
    print(f"RSA-2048: ~10^{fit['rsa']['RSA-2048']['ops']:.0f} ops, ~10^{fit['rsa']['RSA-2048']['years']:.0f} years")
    
    plot_results(results, fit)

if __name__ == "__main__":
    main()

[1/1000] 0.1% | ETA: 0s
[10/1000] 1.0% | ETA: 92s
[20/1000] 2.0% | ETA: 161s
[30/1000] 3.0% | ETA: 274s
[40/1000] 4.0% | ETA: 392s
